<a href="https://colab.research.google.com/github/Ayushman125/Essentials_of_AI/blob/main/EAI_LAB_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Experiment 5




1) Write a python script to determine the size of a sample

>   a)Random

>  b)Cluster-based Sampling

>   c)Strade based Sampling

  write your own funcutions with required parameters.

# Sample Size Calculation Formulas

### 1. Simple Random Sampling
Determines sample size based on a desired confidence level and margin of error for a large or unknown population using Cochran's Formula:
$$n = \frac{Z^2 \cdot p \cdot (1 - p)}{e^2}$$

*Adjusted for a Finite Population ($N$):*
$$n_{adjusted} = \frac{n}{1 + \frac{n - 1}{N}}$$

### 2. Cluster-Based Sampling
Accounts for high intra-cluster correlation by inflating the simple random sample size using the **Design Effect (DEFF)**:
$$n_{cluster} = n \cdot [1 + (m - 1) \cdot \rho]$$
Where:
- $m$ = Average number of individuals per cluster.
- $\rho$ (rho) = Intra-cluster correlation coefficient (ICC).

### 3. Stratified Sampling
Allocates a total sample size across distinct strata using **Proportional Allocation**:
$$n_h = n \cdot \left( \frac{N_h}{N} \right)$$
Where:
- $N_h$ = Population size of stratum $h$.
- $N$ = Total population size ($N = \sum N_h$).
- $n$ = Total target sample size.

In [1]:
import math

# Standard Normal Z-scores for common confidence levels
Z_DICT = {
    80: 1.282,
    85: 1.440,
    90: 1.645,
    95: 1.960,
    99: 2.576
}

def get_z_score(confidence_level):
    """Returns the Z-score for standard confidence levels or calculates approximation."""
    if confidence_level in Z_DICT:
        return Z_DICT[confidence_level]
    else:
        # Fallback approximation for custom levels
        alpha = 1 - (confidence_level / 100)
        return round(math.sqrt(2) * math.erfinv(1 - alpha), 3)

def random_sample_size(pop_size=None, confidence_level=95, margin_of_error=0.05, proportion=0.5):
    """
    Calculates sample size for Simple Random Sampling using Cochran's formula.
    """
    z = get_z_score(confidence_level)
    e = margin_of_error
    p = proportion

    # Infinite population formula
    n0 = (z**2 * p * (1 - p)) / (e**2)

    # Adjust for finite population if N is provided
    if pop_size and pop_size > 0:
        n = n0 / (1 + ((n0 - 1) / pop_size))
    else:
        n = n0

    return math.ceil(n)


def cluster_sample_size(base_sample_size, cluster_size, icc):
    """
    Calculates sample size for Cluster-Based Sampling by applying Design Effect (DEFF).
    """
    deff = 1 + (cluster_size - 1) * icc
    total_sample_size = base_sample_size * deff
    num_clusters = math.ceil(total_sample_size / cluster_size)
    adjusted_total_sample = num_clusters * cluster_size

    return adjusted_total_sample, num_clusters, round(deff, 3)


def stratified_sample_size(total_sample_size, strata_populations):
    """
    Calculates sample sizes for each stratum using proportional allocation.
    """
    total_pop = sum(strata_populations.values())
    strata_sample_sizes = {}

    for stratum, pop in strata_populations.items():
        n_h = total_sample_size * (pop / total_pop)
        strata_sample_sizes[stratum] = math.ceil(n_h)

    return strata_sample_sizes, total_pop


def main():
    print("=== SAMPLE SIZE CALCULATOR ===")
    print("Select Sampling Method:")
    print("1) Simple Random Sampling")
    print("2) Cluster-based Sampling")
    print("3) Stratified-based Sampling")

    choice = input("\nEnter choice (1, 2, or 3): ").strip()

    if choice == '1':
        print("\n--- Simple Random Sampling ---")
        pop_str = input("Enter Total Population Size (press Enter if unknown/infinite): ").strip()
        pop_size = int(pop_str) if pop_str else None

        conf = float(input("Enter Confidence Level % (e.g., 90, 95, 99) [Default 95]: ") or 95)
        moe = float(input("Enter Margin of Error (e.g., 0.05 for 5%) [Default 0.05]: ") or 0.05)
        p = float(input("Enter Estimated Proportion (e.g., 0.5 for 50%) [Default 0.5]: ") or 0.5)

        n = random_sample_size(pop_size, conf, moe, p)
        print(f"\n>> Required Random Sample Size: {n} units")

    elif choice == '2':
        print("\n--- Cluster-based Sampling ---")
        base_n = int(input("Enter baseline Simple Random Sample Size (or calculate via Option 1 first): "))
        cluster_size = int(input("Enter average number of subjects per cluster: "))
        icc = float(input("Enter Intra-cluster Correlation Coefficient (ICC / rho, e.g., 0.02 to 0.05): "))

        total_n, clusters, deff = cluster_sample_size(base_n, cluster_size, icc)
        print(f"\n>> Design Effect (DEFF): {deff}")
        print(f">> Total Sample Size Required: {total_n} units")
        print(f">> Total Clusters to Sample: {clusters} clusters (with {cluster_size} subjects each)")

    elif choice == '3':
        print("\n--- Stratified-based Sampling ---")
        total_n = int(input("Enter Total Target Sample Size: "))
        num_strata = int(input("Enter number of Strata: "))

        strata_pops = {}
        for i in range(num_strata):
            name = input(f"  Enter name for Stratum {i+1}: ") or f"Stratum_{i+1}"
            pop = int(input(f"  Enter population size for {name}: "))
            strata_pops[name] = pop

        allocations, total_pop = stratified_sample_size(total_n, strata_pops)

        print(f"\n>> Total Population Across Strata: {total_pop}")
        print(">> Proportional Sample Allocation per Stratum:")
        for stratum, n_h in allocations.items():
            print(f"   - {stratum}: {n_h} units")

    else:
        print("Invalid selection.")

# Execute interactive menu
if __name__ == "__main__":
    main()

=== SAMPLE SIZE CALCULATOR ===
Select Sampling Method:
1) Simple Random Sampling
2) Cluster-based Sampling
3) Stratified-based Sampling

Enter choice (1, 2, or 3): 1

--- Simple Random Sampling ---
Enter Total Population Size (press Enter if unknown/infinite): 1000
Enter Confidence Level % (e.g., 90, 95, 99) [Default 95]: 90
Enter Margin of Error (e.g., 0.05 for 5%) [Default 0.05]: 0.05
Enter Estimated Proportion (e.g., 0.5 for 50%) [Default 0.5]: 0.5

>> Required Random Sample Size: 214 units


2. For a given population of size 'n' (generate using random numbers), select 'k'(k<<<n) sample of size n' (n'<n) and find the best sample among 'k' sample by measuring goodness (MOCT,MOS) of samples.

# Sample Goodness Evaluation Formulas

To determine the "best" sample out of $k$ drawn samples, we calculate the absolute relative error between each sample's statistical properties and the underlying population properties.

### 1. Measures of Central Tendency (MOCT) Error
Evaluates how well the sample mean ($\bar{x}$) represents the population mean ($\mu$):
$$\text{Error}_{\text{MOCT}} = \left| \frac{\bar{x} - \mu}{\mu} \right|$$

### 2. Measures of Spread (MOS) Error
Evaluates how well the sample standard deviation ($s$) represents the population standard deviation ($\sigma$):
$$\text{Error}_{\text{MOS}} = \left| \frac{s - \sigma}{\sigma} \right|$$

### 3. Total Goodness Score (Loss Metric)
Combines MOCT and MOS into a single goodness error metric (where lower values indicate a better, more representative sample):
$$\text{Score}_{\text{Goodness}} = w_1 \cdot \text{Error}_{\text{MOCT}} + w_2 \cdot \text{Error}_{\text{MOS}}$$
*(By default, equal weights $w_1 = 0.5$ and $w_2 = 0.5$ are applied).*

In [2]:
import numpy as np
import pandas as pd

def generate_population(N, mean=100, std_dev=15, seed=42):
    """Generates a synthetic population of size N following a normal distribution."""
    np.random.seed(seed)
    population = np.random.normal(loc=mean, scale=std_dev, size=N)
    return population


def calculate_moct_mos(data):
    """Calculates Measures of Central Tendency (Mean) and Measures of Spread (Std Dev)."""
    moct = np.mean(data)
    mos = np.std(data, ddof=1) if len(data) > 1 else 0.0  # Sample standard deviation
    return moct, mos


def evaluate_samples(population, k, sample_size, w_moct=0.5, w_mos=0.5):
    """
    Selects k samples of size sample_size from population and finds the best sample
    based on MOCT and MOS goodness metrics.
    """
    # Population benchmark statistics
    pop_moct, pop_mos = calculate_moct_mos(population)

    results = []
    samples_data = []

    for i in range(k):
        # Draw random sample without replacement
        sample = np.random.choice(population, size=sample_size, replace=False)
        samples_data.append(sample)

        sample_moct, sample_mos = calculate_moct_mos(sample)

        # Calculate Relative Errors
        moct_error = abs(sample_moct - pop_moct) / abs(pop_moct)
        mos_error = abs(sample_mos - pop_mos) / abs(pop_mos)

        # Combined Goodness Error (Lower is better)
        total_goodness_error = (w_moct * moct_error) + (w_mos * mos_error)

        results.append({
            'Sample_ID': i + 1,
            'Sample_Mean (MOCT)': round(sample_moct, 4),
            'Sample_StdDev (MOS)': round(sample_mos, 4),
            'MOCT_Error': round(moct_error, 6),
            'MOS_Error': round(mos_error, 6),
            'Goodness_Score': round(total_goodness_error, 6)
        })

    results_df = pd.DataFrame(results)

    # Best sample has the minimum combined goodness error
    best_idx = results_df['Goodness_Score'].idxmin()
    best_sample_info = results_df.loc[best_idx]
    best_sample_data = samples_data[best_idx]

    benchmark_info = {
        'Pop_Mean (MOCT)': round(pop_moct, 4),
        'Pop_StdDev (MOS)': round(pop_mos, 4)
    }

    return results_df, best_sample_info, best_sample_data, benchmark_info


def main():
    print("=== SAMPLE GOODNESS EVALUATOR (MOCT & MOS) ===")

    # User Inputs
    N = int(input("Enter Population Size N (e.g., 10000): ") or 10000)
    k = int(input("Enter number of samples k to select (e.g., 10): ") or 10)
    n_prime = int(input(f"Enter sample size n' (n' < {N}, e.g., 100): ") or 100)

    if n_prime >= N:
        print("Error: Sample size n' must be strictly less than Population size N.")
        return

    # Generate synthetic population
    print("\nGenerating population...")
    population = generate_population(N)

    # Evaluate samples
    results_df, best_sample_info, best_sample_data, benchmark_info = evaluate_samples(
        population=population,
        k=k,
        sample_size=n_prime
    )

    # Display Results
    print("\n--- POPULATION BENCHMARKS ---")
    print(f"Population Mean (MOCT)   : {benchmark_info['Pop_Mean (MOCT)']}")
    print(f"Population StdDev (MOS) : {benchmark_info['Pop_StdDev (MOS)']}")

    print("\n--- SAMPLES EVALUATION TABLE ---")
    print(results_df.to_string(index=False))

    print("\n" + "="*45)
    print(f"   BEST SAMPLE: Sample #{int(best_sample_info['Sample_ID'])}")
    print("="*45)
    print(f"Mean (MOCT)       : {best_sample_info['Sample_Mean (MOCT)']}")
    print(f"StdDev (MOS)      : {best_sample_info['Sample_StdDev (MOS)']}")
    print(f"MOCT Error        : {best_sample_info['MOCT_Error'] * 100:.4f}%")
    print(f"MOS Error         : {best_sample_info['MOS_Error'] * 100:.4f}%")
    print(f"Combined Loss     : {best_sample_info['Goodness_Score']}")

# Execute interactive session
if __name__ == "__main__":
    main()

=== SAMPLE GOODNESS EVALUATOR (MOCT & MOS) ===
Enter Population Size N (e.g., 10000): 1000
Enter number of samples k to select (e.g., 10): 10
Enter sample size n' (n' < 1000, e.g., 100): 500

Generating population...

--- POPULATION BENCHMARKS ---
Population Mean (MOCT)   : 100.29
Population StdDev (MOS) : 14.6882

--- SAMPLES EVALUATION TABLE ---
 Sample_ID  Sample_Mean (MOCT)  Sample_StdDev (MOS)  MOCT_Error  MOS_Error  Goodness_Score
         1            100.5419              14.8314    0.002512   0.009746        0.006129
         2            100.4546              14.6491    0.001642   0.002663        0.002152
         3            100.2891              15.0052    0.000009   0.021578        0.010793
         4            100.8111              14.2483    0.005197   0.029952        0.017574
         5            100.6285              14.9931    0.003376   0.020756        0.012066
         6            100.2288              15.2195    0.000610   0.036168        0.018389
         7   